<!-- nav -->
[← 5 · Operators](05_Operators.ipynb) · [Home](../../README.md) · [7 · Bindings →](07_Bindings.ipynb)

# 6 · Calls, members and subscripts

Tutorial 5 built expressions from operators. Most code also reaches into values: it reads a property (`x.celsius`), an
item (`x[0]`) or calls a function (`f(x)`). Most languages write these as postfix operations on a value, and each
dialect models them with kinds of its own, which this notebook builds in the builders' calls, beside each language's
text. Where two languages mean different things by the same text, or one has no counterpart for another's operation,
the translators say so rather than guess.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `variable` writes a
name in the dialects whose names are the framework's `variable` (tutorial 4), as tutorial 5 does, and `translate`
translates an expression from one dialect to another, as case study 14 does. Python's notebook also reads Python's text,
with `Python3.Text.FromText`, which only Python can (tutorial 3); here each Python model is built.

In [1]:
import assert from "node:assert/strict";
import { Repr } from "@mbse/schemas/Framework";
import * as Translators from "@mbse/expressions/Translators";
import { Simplifiers, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as C from "@mbse/expressions/C/Expressions";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as Excel from "@mbse/expressions/Excel/Expressions";
import * as Matlab from "@mbse/expressions/MATLAB/Expressions";
import { Terms } from "@mbse/expressions/Framework";
import { dialects, show } from "./toolkit.js";

const store = dialects();
const [S, Term] = [store.Expressions, store.Expressions.Framework.Term];
const amd64 = store.Instances.Expressions.SystemV.AMD64;
const variable = (name: string) => (d: any) => d.variable((v: any) => v.name(name));
const languages: Record<string, [any, any]> = {
  Python3: [Python3.Expressions.DIALECT, Python3.Text], C: [C.Expressions.DIALECT, C.Text],
  SystemVerilog: [SystemVerilog.Expressions.DIALECT, SystemVerilog.Text],
  Excel: [Excel.Expressions.DIALECT, Excel.Text], MATLAB: [Matlab.Expressions.DIALECT, Matlab.Text],
};

/** `expression`, of the dialect `source`, translated to `target`, written as a term. */
function translate(expression: unknown, source: any, target: any): any {
  return store.transaction(() => Term(Translators.between(source, target).forward(expression)));
}

<a id="T06.1"></a>
## 06.1 · A property

Patterns reads a property with an operation, `get`: `get(x, 'celsius')` is a `call` whose `callable` is `get`, its
arguments the value and the property's name, a literal. Each language has a kind of its own for `x.celsius`: Python3's
`attribute` (`ast`'s `Attribute`, its `attr` a string), TypeScript5's `memberexpression` (ESTree's, its `property` an
`identifier` term), C's `memberexpression` (its `operator` `.` or `->`, its `member` a name), SystemVerilog's
`memberexpression` (its `member` an identifier), and Excel's and MATLAB's `field` (its `name` a string):

In [2]:
const [get, members]: [any, Record<string, [any, any]>] = store.transaction(() => [
  S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c.callable("get").arguments([
    (t: any) => t.patterns(variable("x")),
    (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((a: any) => a.as_scalar("celsius"))))])))),
  {
    Python3: [Python3.Text, S.Python.Python3.Expression((e: any) => e.root(
      (u: any) => u.attribute((a: any) => a.value(variable("x")).attr("celsius"))))],
    TypeScript5: [TypeScript5.Text, S.TypeScript.TypeScript5.Expression((e: any) => e.root(
      (u: any) => u.memberexpression((m: any) => m
        .object((o: any) => o.identifier((i: any) => i.name("x")))
        .property((p: any) => p.identifier((i: any) => i.name("celsius"))))))],
    C: [C.Text, S.C.Expressions.Expression((e: any) => e.root(
      (u: any) => u.memberexpression((m: any) => m
        .object(variable("x")).operator(".").member((n: any) => n.identifier((i: any) => i.spelling("celsius")))),
    ).abi([amd64]))],
    "C, through a pointer": [C.Text, S.C.Expressions.Expression((e: any) => e.root(
      (u: any) => u.memberexpression((m: any) => m
        .object(variable("x")).operator("->").member((n: any) => n.identifier((i: any) => i.spelling("celsius")))),
    ).abi([amd64]))],
    SystemVerilog: [SystemVerilog.Text, S.SystemVerilog.Expressions.Expression((e: any) => e.root(
      (u: any) => u.memberexpression((m: any) => m
        .value((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("x")))))
        .member((i: any) => i.spelling("celsius")))))],
    Excel: [Excel.Text, S.Excel.Expressions.Expression((e: any) => e.root(
      (u: any) => u.field((f: any) => f.value(variable("x")).name("celsius"))))],
    MATLAB: [Matlab.Text, S.MATLAB.Expressions.Expression((e: any) => e.root(
      (u: any) => u.field((f: any) => f.value(variable("x")).name("celsius"))))],
  },
]);
console.log("Patterns".padEnd(21), show(get));
for (const [language, [text, member]] of Object.entries(members)) {
  console.log(language.padEnd(21), text.ToText(member).padEnd(12), Terms.form(Terms.term(member)).kind);
}
assert.deepEqual(Object.values(members).map(([text, member]) => text.ToText(member)), [
  "x.celsius", "x.celsius", "x.celsius", "x->celsius", "x.celsius", "=x.celsius", "x.celsius"]);

Patterns              get(x, 'celsius')


Python3               x.celsius    attribute


TypeScript5           x.celsius    memberexpression


C                     x.celsius    memberexpression


C, through a pointer  x->celsius   memberexpression


SystemVerilog         x.celsius    memberexpression


Excel                 =x.celsius   field


MATLAB                x.celsius    field


The translators write `get` as each language's own member kind, the same terms as built above (TypeScript5 has no
translator yet). An object's related objects, `entries(x, 'loggers')` (case study 11), are Patterns' alone: they are
read through the store's relations, which no language's member reaches, and every translator says so:

In [3]:
const entries = store.transaction(() => S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.call(
  (c: any) => c.callable("entries").arguments([(t: any) => t.patterns(variable("x")),
    (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((a: any) => a.as_scalar("loggers"))))])))));
for (const [language, [dialect, text]] of Object.entries(languages)) {
  const translated = translate(get, E.DIALECT, dialect);
  assert.ok(Terms.same(translated, members[language]![1]));
  try {
    console.log(language.padEnd(14), text.ToText(translated).padEnd(12), text.ToText(translate(entries, E.DIALECT, dialect)));
  } catch (error) {
    console.log(language.padEnd(14), text.ToText(translated).padEnd(12), (error as Error).message);
  }
}

Python3        x.celsius    Patterns call 'entries' has no Python3 counterpart


C              x.celsius    Patterns call 'entries' has no C counterpart


SystemVerilog  x.celsius    Patterns call 'entries' has no SystemVerilog counterpart


Excel          =x.celsius   Patterns call 'entries' has no Excel counterpart


MATLAB         x.celsius    Patterns call 'entries' has no MATLAB counterpart


<a id="T06.2"></a>
## 06.2 · An item

Patterns' `item(x, 0)` is a collection's first item: it counts from 0, as Python, TypeScript, C and SystemVerilog do.
Each writes `x[0]`, each with a kind of its own: Python3's `subscript`, TypeScript5's `memberexpression` with
`computed` set (in JavaScript, `x[0]` and `x.celsius` are one operation), C's `subscriptexpression` and SystemVerilog's
`indexexpression`. Excel and MATLAB count from 1: Excel has no subscript, but a function, `INDEX`; MATLAB's `index` is
`x(i)`, written as a call is. So their item 0 is `INDEX(x, 0 + 1)` and `x(0 + 1)`:

In [4]:
const [item, items]: [any, Record<string, [any, any]>] = store.transaction(() => [
  S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c.callable("item").arguments([
    (t: any) => t.patterns(variable("x")),
    (t: any) => t.patterns((d: any) => d.literal((r: any) => r.value((a: any) => a.as_scalar(0n))))])))),
  {
    Python3: [Python3.Text, S.Python.Python3.Expression((e: any) => e.root((u: any) => u.subscript((s: any) => s
      .value(variable("x")).slice((d: any) => d.literal((r: any) => r.value((a: any) => a.as_scalar(0n)).text("0").base(10))))))],
    TypeScript5: [TypeScript5.Text, S.TypeScript.TypeScript5.Expression((e: any) => e.root(
      (u: any) => u.memberexpression((m: any) => m
        .object((o: any) => o.identifier((i: any) => i.name("x"))).computed(true)
        .property((p: any) => p.literal((r: any) => r.value((a: any) => a.as_scalar(0)).text("0"))))))],
    C: [C.Text, S.C.Expressions.Expression((e: any) => e.root((u: any) => u.subscriptexpression((s: any) => s
      .object(variable("x")).indices([(d: any) => d.literal((r: any) => r.value((a: any) => a.as_scalar(0n)))]))).abi([amd64]))],
    SystemVerilog: [SystemVerilog.Text, S.SystemVerilog.Expressions.Expression((e: any) => e.root(
      (u: any) => u.indexexpression((s: any) => s
        .value((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("x")))))
        .index((d: any) => d.literal((r: any) => r.value((a: any) => a.as_scalar(0n)))))))],
    Excel: [Excel.Text, S.Excel.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c
      .callable("INDEX").arguments([
        (t: any) => t.excel(variable("x")),
        (t: any) => t.excel((d: any) => d.infix((i: any) => i
          .left((l: any) => l.literal((r: any) => r.value((a: any) => a.as_scalar(0n)))).operator("+")
          .right((l: any) => l.literal((r: any) => r.value((a: any) => a.as_scalar(1n))))))]))))],
    MATLAB: [Matlab.Text, S.MATLAB.Expressions.Expression((e: any) => e.root((u: any) => u.index((s: any) => s
      .value(variable("x")).index((d: any) => d.binary((b: any) => b
        .left((l: any) => l.literal((r: any) => r.value((a: any) => a.as_scalar(0n)))).operator("+")
        .right((l: any) => l.literal((r: any) => r.value((a: any) => a.as_scalar(1n)))))))))],
  },
]);
console.log("Patterns".padEnd(14), show(item), "of [3.5, 4.5] is", Simplifiers.OfAny(item, { x: [3.5, 4.5] }).value);
for (const [language, [text, built]] of Object.entries(items)) {
  console.log(language.padEnd(14), text.ToText(built).padEnd(17), Terms.form(Terms.term(built)).kind);
}
assert.deepEqual(Object.values(items).map(([text, built]) => text.ToText(built)), [
  "x[0]", "x[0]", "x[0]", "x[0]", "=INDEX(x, 0 + 1)", "x(0 + 1)"]);

Patterns       item(x, 0) of [3.5, 4.5] is 3.5


Python3        x[0]              subscript


TypeScript5    x[0]              memberexpression


C              x[0]              subscriptexpression


SystemVerilog  x[0]              indexexpression


Excel          =INDEX(x, 0 + 1)  call


MATLAB         x(0 + 1)          index


Translating `item` writes out each language's counting, so that item 0 is the same item everywhere. C has a subscript,
but over arrays and pointers, which hold no length; translating Patterns' collections to C is a capability still to
come ([CAP-102: collections translated to LaTeX and C and C++](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-102.md)),
and its translator refuses `item` by name rather than write a subscript that means something else:

In [5]:
for (const [language, [dialect, text]] of Object.entries(languages)) {
  try {
    const translated = translate(item, E.DIALECT, dialect);
    console.log(language.padEnd(14), text.ToText(translated));
    assert.equal(text.ToText(translated), text.ToText(items[language]![1]));
  } catch (error) {
    console.log(language.padEnd(14), (error as Error).message);
  }
}

Python3        x[0]


C              Patterns call 'item' has no C counterpart


SystemVerilog  x[0]


Excel          =INDEX(x, 0 + 1)


MATLAB         x(0 + 1)


<a id="T06.3"></a>
## 06.3 · A call

Patterns calls an operation by name: in `max(x)` the `callable` is an attribute, the string `max`, never a value, and
validation refuses a name its dialect does not define (tutorial 5, 05.6). Excel and MATLAB call functions the same way,
with the framework's `call`, so `=MAX(x)` and `max(x)` read back as Patterns' `max(x)`:

In [6]:
const [biggest, unknown, excel, matlab] = store.transaction(() => [
  S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c.callable("max")
    .arguments([(t: any) => t.patterns(variable("x"))])))),
  S.Patterns.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c.callable("f")
    .arguments([(t: any) => t.patterns(variable("x"))])))),
  S.Excel.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c.callable("MAX")
    .arguments([(t: any) => t.excel(variable("x"))])))),
  S.MATLAB.Expressions.Expression((e: any) => e.root((u: any) => u.call((c: any) => c.callable("max")
    .arguments([(t: any) => t.matlab(variable("x"))])))),
]);
const biggestForm = Terms.form(Terms.term(biggest));
console.log(biggestForm.kind, biggestForm.attributes.get("callable"), biggestForm.arguments.length);
console.log(...E.DIALECT.validate(unknown, ["x"]));
for (const [language, text, built] of [["Excel", Excel.Text, excel], ["MATLAB", Matlab.Text, matlab]] as [string, any, any][]) {
  const dialect = languages[language]![0];
  console.log(language.padEnd(7), text.ToText(built).padEnd(8), Terms.form(Terms.term(built)).kind, show(translate(built, dialect, E.DIALECT)));
}

call max 1


'f' is not an operation Patterns defines


Excel   =MAX(x)  call max(x)


MATLAB  max(x)   call max(x)


Python3's `call` applies any term, its `func`: a name, an attribute, another call's result. So do TypeScript5's
`callexpression` (its `callee`) and C's (its `function`). Python's `max(x)` names a builtin that Patterns has, and reads
back as Patterns' `max(x)`; `f(x)` calls a value, which Patterns cannot, and its translator says so:

In [7]:
const calls: Record<string, [any, any]> = store.transaction(() => ({
  Python3: [Python3.Text, S.Python.Python3.Expression((e: any) => e.root((u: any) => u.call((c: any) => c
    .func(variable("f")).args([variable("x")]))))],
  "Python3 builtin": [Python3.Text, S.Python.Python3.Expression((e: any) => e.root((u: any) => u.call((c: any) => c
    .func(variable("max")).args([variable("x")]))))],
  TypeScript5: [TypeScript5.Text, S.TypeScript.TypeScript5.Expression((e: any) => e.root(
    (u: any) => u.callexpression((c: any) => c
      .callee((o: any) => o.identifier((i: any) => i.name("f")))
      .arguments([(o: any) => o.identifier((i: any) => i.name("x"))]))))],
  C: [C.Text, S.C.Expressions.Expression((e: any) => e.root((u: any) => u.callexpression((c: any) => c
    .function(variable("f")).arguments([variable("x")]))).abi([amd64]))],
}));
for (const [language, [text, built]] of Object.entries(calls)) {
  const form = Terms.form(Terms.term(built));
  console.log(language.padEnd(16), text.ToText(built).padEnd(7), form.kind, form.arguments.length);
}
const python = languages.Python3![0];
console.log(show(translate(calls["Python3 builtin"]![1], python, E.DIALECT)));
try {
  translate(calls.Python3![1], python, E.DIALECT);
} catch (error) {
  console.log((error as Error).message);
}

Python3          f(x)    call 2


Python3 builtin  max(x)  call 2


TypeScript5      f(x)    callexpression 2


C                f(x)    callexpression 2


max(x)


Python3 call has no Patterns counterpart


SystemVerilog calls a system function by name, `$clog2(x)`, a `systemcall`, and an array's methods as calls of a member,
`x.sum()`, a `callexpression` whose `callee` is a `memberexpression`. A method that Patterns has reads back as its
operation; a system function has no counterpart, and its translator says so:

In [8]:
const [system, method] = store.transaction(() => [
  S.SystemVerilog.Expressions.Expression((e: any) => e.root((u: any) => u.systemcall((c: any) => c
    .name("$clog2").arguments([
      (t: any) => t.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("x"))))])))),
  S.SystemVerilog.Expressions.Expression((e: any) => e.root((u: any) => u.callexpression((c: any) => c
    .callee((t: any) => t.memberexpression((m: any) => m
      .value((d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling("x")))))
      .member((i: any) => i.spelling("sum"))))
    .arguments([])))),
]);
for (const built of [system, method]) {
  let back: string;
  try {
    back = show(translate(built, SystemVerilog.Expressions.DIALECT, E.DIALECT));
  } catch (error) {
    back = (error as Error).message;
  }
  console.log(SystemVerilog.Text.ToText(built).padEnd(9), Terms.form(Terms.term(built)).kind.padEnd(14), back);
}

$clog2(x) systemcall     SystemVerilog systemcall has no Patterns counterpart


x.sum()   callexpression sum(x)


<a id="T06.4"></a>
## 06.4 · A chain

Postfix operations nest left to right: in `a.b.c(d)[e:f]` the subscript applies to the call's result, the call to the
attribute `c`, and `c` to the attribute `b` of `a`. So the outermost term is the last operation written, and the first
name written, `a`, is the deepest. `Terms.walk` reads the tree parents first; a slice, `e:f`, is a term of its own
(Python3's `slice`, its `lower`, `upper` and `step`), the subscript's `slice`:

In [9]:
const chain = store.transaction(() => S.Python.Python3.Expression((e: any) => e.root((u: any) => u.subscript((s: any) => s
  .value((d: any) => d.call((c: any) => c
    .func((f: any) => f.attribute((a: any) => a
      .value((b: any) => b.attribute((a: any) => a.value(variable("a")).attr("b"))).attr("c")))
    .args([variable("d")])))
  .slice((d: any) => d.slice((r: any) => r.lower(variable("e")).upper(variable("f"))))))));
console.log(Python3.Text.ToText(chain));
assert.equal(Python3.Text.ToText(chain), "a.b.c(d)[e:f]");
for (const term of Terms.walk(chain)) {
  const form = Terms.form(term);
  console.log(form.kind.padEnd(10), [...form.attributes].map(([name, value]) => `${name}=${value}`).join(", "));
}

a.b.c(d)[e:f]


subscript  


call       


attribute  attr=c


attribute  attr=b


variable   name=a


variable   name=d


slice      


variable   name=e


variable   name=f


<a id="T06.5"></a>
## 06.5 · Helpers are parts, not terms

Python's `f(x, y=1)` passes `y=1` by keyword. A keyword is not a term: `y=1` is no expression of its own, and nothing
evaluates it. It is a helper, Python3's `keyword` (`ast`'s), a value object a call holds in its `keywords`, its `arg` a
name and its `value` a term. A form reads a helper as a `Terms.Part` among its term's attributes: its kind, its own
attributes, and how many of the term's arguments are its terms, which follow the term's own subterms and variadic
arguments. So the call's arguments are `f`, `x`, then the keyword's value, `1`, and `make` writes the form back:

In [10]:
const [keyword, made] = store.transaction(() => {
  const keyword = S.Python.Python3.Expression((e: any) => e.root((u: any) => u.call((c: any) => c
    .func(variable("f")).args([variable("x")])
    .keywords([(k: any) => k.arg("y").value((d: any) => d.literal(
      (r: any) => r.value((a: any) => a.as_scalar(1n)).text("1").base(10)))]))));
  return [keyword, Term(Python3.Expressions.DIALECT.make(Terms.form(Terms.term(keyword))))];
});
const form = Terms.form(Terms.term(keyword));
const [part] = form.attributes.get("keywords") as Terms.Part[];
console.log(Python3.Text.ToText(keyword));
console.log(form.kind, Repr.repr(part));
console.log(form.arguments.map((argument) => Python3.Text.ToText(argument)).join(", "));
assert.ok(part!.kind === "keyword" && part!.attributes.get("arg") === "y" && part!.size === 1);
assert.ok(Terms.same(made, keyword));

f(x, y=1)


call Part(kind='keyword', attributes={'arg': 'y'}, size=1)


f, x, 1


## What you learned

- **Each language reaches into values its own way.** A property is Python3's `attribute`, TypeScript5's, C's and
  SystemVerilog's `memberexpression`, Excel's and MATLAB's `field`; Patterns reads it with its operation `get`.
- **An item's index is the language's.** Patterns, Python, TypeScript, C and SystemVerilog count from 0; Excel and
  MATLAB from 1, and the translators write the difference out.
- **A call names an operation, or applies a value.** Patterns, Excel and MATLAB call by name; Python3, TypeScript5, C
  and SystemVerilog apply a term. What has no counterpart, a call of a value in Patterns, Patterns' `item` in C,
  Patterns' `entries` anywhere else, is refused by name, never guessed.
- **Postfix operations nest left to right**, the last written outermost.
- **A helper is a part, not a term**: a form keeps its structure in its attributes and its terms in its arguments.

Next, tutorial 7 binds names: `let`, lambdas, quantifiers and comprehensions.

<!-- nav -->
[← 5 · Operators](05_Operators.ipynb) · [Home](../../README.md) · [7 · Bindings →](07_Bindings.ipynb)